# `sqlglot.parse_one` paso a paso

Qué hace el traductor del gateway con el SQL que llega de un cliente BI:
**texto → tokens → árbol (AST) → navegación → reescritura → texto en otro dialecto**.

No necesita el gateway corriendo. Solo `sqlglot` (y `pandas` para ver tablas).

In [1]:
# %pip install sqlglot pandas    # si el kernel no los tiene
import sqlglot
import sqlglot.expressions as exp
import pandas as pd

print("sqlglot", sqlglot.__version__)

sqlglot 30.21.0


In [2]:
q = """
SELECT metric_time__month, event__platform, MEASURE(item_views) AS vistas
FROM semantic.events
WHERE event__platform IN ('ios','android') AND metric_time >= DATE '2024-01-01'
GROUP BY 1, 2 ORDER BY 1
"""

## Paso 1 · Tokenizar
El dialecto `postgres` corta el texto en piezas con un tipo cada una. `parse_one` lo hace internamente; aquí lo llamamos aparte para verlo.

In [3]:
dialecto = sqlglot.Dialect.get_or_raise("postgres")
tokens = dialecto.tokenize(q)
pd.DataFrame([(t.token_type.name, t.text) for t in tokens], columns=["tipo", "texto"])

,tipo,texto
0,SELECT,SELECT
1,VAR,metric_time__month
2,COMMA,","
3,VAR,event__platform
4,COMMA,","
5,VAR,MEASURE
6,L_PAREN,(
7,VAR,item_views
8,R_PAREN,)
9,ALIAS,AS


## Paso 2 · Parsear: tokens → árbol
Es exactamente la línea del traductor: `sqlglot.parse_one(body, read=self.read)` con `read="postgres"`.

In [4]:
tree = sqlglot.parse_one(q, read="postgres")
print("Tipo de la raíz:", type(tree).__name__)   # el traductor exige exp.Select
print("¿Es SELECT?   :", isinstance(tree, exp.Select))

Tipo de la raíz: Select
¿Es SELECT?   : True


In [5]:
# Partes de la consulta: cada una es un argumento del nodo Select
for clave, valor in tree.args.items():
    if valor:
        tipo = f"list[{len(valor)}]" if isinstance(valor, list) else type(valor).__name__
        print(f"{clave:12} -> {tipo}")

expressions  -> list[3]
from_        -> From
where        -> Where
group        -> Group
order        -> Order


In [6]:
# El árbol completo (repr) — largo, pero muestra la estructura real
print(repr(tree))

Select(
  expressions=[
    Column(
      this=Identifier(this=metric_time__month, quoted=False)),
    Column(
      this=Identifier(this=event__platform, quoted=False)),
    Alias(
      this=Anonymous(
        this=MEASURE,
        expressions=[
          Column(
            this=Identifier(this=item_views, quoted=False))]),
      alias=Identifier(this=vistas, quoted=False))],
  from_=From(
    this=Table(
      this=Identifier(this=dbt_glue_lab, quoted=False),
      db=Identifier(this=semantic, quoted=False))),
  where=Where(
    this=And(
      this=In(
        this=Column(
          this=Identifier(this=event__platform, quoted=False)),
        expressions=[
          Literal(this='ios', is_string=True),
          Literal(this='android', is_string=True)]),
      expression=GTE(
        this=Column(
          this=Identifier(this=metric_time, quoted=False)),
        expression=Cast(
          this=Literal(this='2024-01-01', is_string=True),
          to=DataType(this=DType.DATE, nes

In [7]:
# Todos los nodos, con su profundidad: así 've' sqlglot la consulta
filas = []
for nodo in tree.walk():
    profundidad = 0
    p = nodo.parent
    while p is not None:
        profundidad, p = profundidad + 1, p.parent
    filas.append(("    " * profundidad + type(nodo).__name__, nodo.sql(dialect="postgres")[:60]))
pd.DataFrame(filas, columns=["nodo", "sql del nodo"])

,nodo,sql del nodo
0,Select,"SELECT metric_time__month, event__platform, ME..."
1,Column,metric_time__month
2,Column,event__platform
3,Alias,MEASURE(item_views) AS vistas
4,From,FROM semantic.dbt_glue_lab
5,Where,"WHERE event__platform IN ('ios', 'android') AN..."
6,Group,"GROUP BY 1, 2"
7,Order,ORDER BY 1
8,Identifier,metric_time__month
9,Identifier,event__platform


## Paso 3 · Navegar el SELECT
El traductor recorre `tree.expressions`. Cada columna puede venir con alias y envuelta en una función.

In [8]:
AGREGADOS = (exp.Sum, exp.Max, exp.Min, exp.Avg)

for proyeccion in tree.expressions:
    alias = proyeccion.alias if isinstance(proyeccion, exp.Alias) else None
    nodo = proyeccion.unalias()                 # quita el 'AS vistas'
    envoltorio = None
    if isinstance(nodo, AGREGADOS):             # SUM(x), MAX(x)...
        envoltorio, nodo = type(nodo).__name__, nodo.this
    elif isinstance(nodo, exp.Anonymous) and nodo.name.upper() in ("MEASURE", "AGG"):
        envoltorio, nodo = nodo.name, nodo.expressions[0]   # sqlglot no conoce MEASURE: es 'Anonymous'
    print(f"{proyeccion.sql():35} | envoltorio={envoltorio!s:8} | columna={nodo.name:20} | alias={alias}")

metric_time__month                  | envoltorio=None     | columna=metric_time__month   | alias=None
event__platform                     | envoltorio=None     | columna=event__platform      | alias=None
MEASURE(item_views) AS vistas       | envoltorio=MEASURE  | columna=item_views           | alias=vistas


## Paso 4 · FROM: qué tabla se lee
Así decide `is_semantic` si la consulta va a MetricFlow o al catálogo (DuckDB).

In [9]:
for t in tree.find_all(exp.Table):
    print("esquema:", t.db, "| tabla:", t.name)
print("¿Lee la tabla virtual?", any(t.name.lower() in ("events", "items") for t in tree.find_all(exp.Table)))
print("¿Tiene JOIN?          ", bool(tree.args.get("joins")))
print("¿Tiene subconsultas?  ", tree.find(exp.Subquery) is not None)

esquema: semantic | tabla: dbt_glue_lab
¿Lee la tabla virtual? True
¿Tiene JOIN?           False
¿Tiene subconsultas?   False


## Paso 5 · WHERE: partir por AND y clasificar cada condición

In [10]:
where = tree.args["where"].this
print("Nodo raíz del WHERE:", type(where).__name__)

condiciones = list(where.flatten()) if isinstance(where, exp.And) else [where]
for c in condiciones:
    columnas = [col.name for col in c.find_all(exp.Column)]
    print(f"\n{c.sql()}")
    print("  tipo de condición:", type(c).__name__)
    print("  columnas         :", columnas)
    print("  lado izquierdo   :", type(c.this).__name__, c.this.sql())
    derecho = c.expression if c.expression is not None else c.expressions
    print("  lado derecho     :", repr(derecho)[:120])

Nodo raíz del WHERE: And

event__platform IN ('ios', 'android')
  tipo de condición: In
  columnas         : ['event__platform']
  lado izquierdo   : Column event__platform
  lado derecho     : [Literal(this='ios', is_string=True), Literal(this='android', is_string=True)]

metric_time >= CAST('2024-01-01' AS DATE)
  tipo de condición: GTE
  columnas         : ['metric_time']
  lado izquierdo   : Column metric_time
  lado derecho     : Cast(
  this=Literal(this='2024-01-01', is_string=True),
  to=DataType(this=DType.DATE, nested=False),
  _type=DataType(


In [11]:
# La fecha: DATE '2024-01-01' llega como Cast(Literal AS DATE). Por eso _date() quita el Cast.
fecha = condiciones[1].expression
print(type(fecha).__name__, "->", type(fecha.this).__name__, repr(fecha.this.this))
# Esta condición (metric_time >= fecha) el traductor la convierte en start=2024-01-01, no en filtro.

Cast -> Literal '2024-01-01'


## Paso 6 · GROUP BY, ORDER BY, LIMIT

In [12]:
print("GROUP BY:", [g.sql() for g in tree.args["group"].expressions], " <- el traductor lo ignora")
for o in tree.args["order"].expressions:
    print("ORDER BY:", repr(o.this.sql()), "| es ordinal:", isinstance(o.this, exp.Literal), "| DESC:", bool(o.args.get("desc")))
    if isinstance(o.this, exp.Literal):
        print("  ordinal 1 ->", tree.expressions[int(o.this.this) - 1].sql())
print("LIMIT:", tree.args.get("limit"))

GROUP BY: ['1', '2']  <- el traductor lo ignora
ORDER BY: '1' | es ordinal: True | DESC: False
  ordinal 1 -> metric_time__month
LIMIT: None


## Paso 7 · Reescribir el filtro para MetricFlow
`transform` recorre el árbol y reemplaza nodos. Es lo que hace `_metricflow_filter`.

In [13]:
def a_jinja(nodo):
    if isinstance(nodo, exp.Column):
        return exp.Var(this=f"{{{{ Dimension('{nodo.name}') }}}}")
    return nodo

filtro = condiciones[0]
for d in ("postgres", "snowflake", "redshift", "athena"):
    print(f"{d:10} {filtro.transform(a_jinja).sql(dialect=d)}")

postgres   {{ Dimension('event__platform') }} IN ('ios', 'android')
snowflake  {{ Dimension('event__platform') }} IN ('ios', 'android')
redshift   {{ Dimension('event__platform') }} IN ('ios', 'android')
athena     {{ Dimension('event__platform') }} IN ('ios', 'android')


## Paso 8 · Volver a texto en otro dialecto
El mismo árbol se puede escribir como SQL de cualquier dialecto (útil para ver diferencias de sintaxis).

In [14]:
for d in ("postgres", "snowflake", "athena", "databricks"):
    print(f"--- {d}")
    print(tree.sql(dialect=d, pretty=True))

--- postgres
SELECT
  metric_time__month,
  event__platform,
  MEASURE(item_views) AS vistas
FROM semantic.events
WHERE
  event__platform IN ('ios', 'android')
  AND metric_time >= CAST('2024-01-01' AS DATE)
GROUP BY
  1,
  2
ORDER BY
  1
--- snowflake
SELECT
  metric_time__month,
  event__platform,
  MEASURE(item_views) AS vistas
FROM semantic.events
WHERE
  event__platform IN ('ios', 'android')
  AND metric_time >= CAST('2024-01-01' AS DATE)
GROUP BY
  1,
  2
ORDER BY
  1
--- athena
SELECT
  metric_time__month,
  event__platform,
  MEASURE(item_views) AS vistas
FROM semantic.events
WHERE
  event__platform IN ('ios', 'android')
  AND metric_time >= CAST('2024-01-01' AS DATE)
GROUP BY
  1,
  2
ORDER BY
  1
--- databricks
SELECT
  metric_time__month,
  event__platform,
  MEASURE(item_views) AS vistas
FROM semantic.events
WHERE
  event__platform IN ('ios', 'android')
  AND metric_time >= CAST('2024-01-01' AS DATE)
GROUP BY
  1,
  2
ORDER BY
  1 NULLS LAST


## Paso 9 · Errores

In [15]:
for malo in ["SELEC x FROM t", "", "SELECT 1; SELECT 2"]:
    try:
        r = sqlglot.parse_one(malo, read="postgres")
        print(f"{malo!r:22} -> {type(r).__name__} (no es Select: el traductor lo rechaza)")
    except sqlglot.errors.ParseError as e:
        print(f"{malo!r:22} -> ParseError: {str(e).splitlines()[0]}")

'SELEC x FROM t'       -> ParseError: Invalid expression / Unexpected token. Line 1, Col: 12.
''                     -> ParseError: No expression was parsed from ''
'SELECT 1; SELECT 2'   -> Block (no es Select: el traductor lo rechaza)


## Paso 10 · Resultado del traductor real (opcional)
Busca sola la carpeta `MetricFlow/` (donde están `semantic_gateway/` y `semantic_manifests/`), aunque el kernel arranque en `notebooks/`.

In [17]:
import os, sys
from pathlib import Path

# Buscar la raíz MetricFlow/ (la carpeta que contiene semantic_gateway/), subiendo desde el directorio actual.
# VS Code y Jupyter suelen arrancar el kernel en la carpeta del notebook (notebooks/).
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "semantic_gateway").is_dir()), None)
if raiz is None:
    print("No encontré semantic_gateway/ en", Path.cwd(), "ni en sus carpetas superiores")
else:
    os.chdir(raiz)                       # para que 'semantic_manifests/...' sea relativo a la raíz
    if str(raiz) not in sys.path:
        sys.path.insert(0, str(raiz))
    print("Raíz del proyecto:", raiz)

    from semantic_gateway.engine import SemanticEngine
    from semantic_gateway.translator import Translator
    engine = SemanticEngine("semantic_manifests/dbt_glue_lab/semantic_manifest.json", "athena")
    sq = Translator(engine.catalog("semantic")).translate(q, dialect="redshift")
    for campo in ("metrics", "group_by", "where", "start", "end", "order_by", "limit", "output"):
        print(f"{campo:9} = {getattr(sq, campo)}")

Raíz del proyecto: /Users/rodrigo/DBT-GLUE/dbt_glue_lab
metrics   = ['item_views']
group_by  = ['metric_time__month', 'event__platform']
where     = ["{{ Dimension('event__platform') }} IN ('ios', 'android')"]
start     = 2024-01-01
end       = None
order_by  = ['metric_time__month']
limit     = None
output    = [('metric_time__month', 'metric_time__month'), ('event__platform', 'event__platform'), ('item_views', 'vistas')]
